In [1]:
import pandas as pd
import numpy as np

credit_df = pd.read_csv("../data/processed/credit_risk_base.csv")

credit_df.head()

,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,OCCUPATION_TYPE,CNT_FAM_MEMBERS,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM,DAYS_EMPLOYED_RATIO
0,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,...,Single / not married,House / apartment,-9461,-637,Laborers,1.0,2.007889,0.121978,0.060749,0.067329
1,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,...,Married,House / apartment,-16765,-1188,Core staff,2.0,4.790750,0.132217,0.027598,0.070862
2,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,...,Single / not married,House / apartment,-19046,-225,Laborers,1.0,2.000000,0.100000,0.050000,0.011814
3,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,...,Civil marriage,House / apartment,-19005,-3039,Laborers,2.0,2.316167,0.219900,0.094941,0.159905
4,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,...,Single / not married,House / apartment,-19932,-3038,Core staff,1.0,4.222222,0.179963,0.042623,0.152418


In [2]:
print("Shape:", credit_df.shape)
print(credit_df["TARGET"].value_counts())
print(credit_df["TARGET"].value_counts(normalize=True) * 100)

Shape: (307511, 22)
TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64


In [3]:
X = credit_df.drop("TARGET", axis=1)
y = credit_df["TARGET"]

In [4]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", len(numeric_cols))
print(list(numeric_cols))

print("Categorical columns:", len(categorical_cols))
print(list(categorical_cols))

Numerical columns: 12
['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_FAM_MEMBERS', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'DAYS_EMPLOYED_RATIO']
Categorical columns: 9
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (246008, 21)
X_test: (61503, 21)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_proba = log_model.predict_proba(X_test)[:, 1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, log_proba))
print(classification_report(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))

Logistic Regression ROC-AUC: 0.6609832448063966
              precision    recall  f1-score   support

           0       0.95      0.61      0.74     56538
           1       0.12      0.62      0.21      4965

    accuracy                           0.61     61503
   macro avg       0.54      0.62      0.48     61503
weighted avg       0.88      0.61      0.70     61503

[[34620 21918]
 [ 1863  3102]]


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_proba))
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

Random Forest ROC-AUC: 0.6439619485038661
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.14      0.00      0.00      4965

    accuracy                           0.92     61503
   macro avg       0.53      0.50      0.48     61503
weighted avg       0.86      0.92      0.88     61503

[[56532     6]
 [ 4964     1]]


In [9]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(log_model, "../models/credit_risk_model.pkl")

print("Logistic Regression credit risk model saved successfully!")

Logistic Regression credit risk model saved successfully!


In [ ]:
# import joblib
# import os

# os.makedirs("../models", exist_ok=True)

# joblib.dump(rf_model, "../models/credit_risk_model.pkl")

# print("Random Forest credit risk model saved successfully!")

Random Forest credit risk model saved successfully!


In [10]:
import joblib

loaded_model = joblib.load("../models/credit_risk_model.pkl")

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[:, 1][0]

print("Prediction:", prediction)
print("Default Probability:", probability)

if probability >= 0.7:
    risk_level = "High Risk"
elif probability >= 0.4:
    risk_level = "Medium Risk"
else:
    risk_level = "Low Risk"

print("Risk Level:", risk_level)

Prediction: 1
Default Probability: 0.5082465563035811
Risk Level: Medium Risk
